In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "global_settings.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)
from src.global_settings import settings


In [ ]:
from pathlib import Path
from typing import Any, Dict, List

import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import pandas as pd
import numpy as np
from pprint import pprint
import matplotlib.pyplot as plt

from src.sae import SparseAutoencoder
from src.sae_adapter import SAEAdapter
from src.features import load_score_data, load_sample_records
from src.utils.model_loader import load_sae
from src.global_settings import settings

In [ ]:
META_DIR = Path(settings.feature_meta_dir)
RECORD_DIR = Path(settings.feature_record_dir)
EXPLANATION_DIR = Path(settings.feature_explanation_dir)

In [ ]:
MODEL = "f3072_k8"
CHECKPOINT = f"{MODEL}_alpha"

LAYER = 21

meta_df = pd.read_csv(settings.text_dataset_path)

scores_data = load_score_data(checkpoint=CHECKPOINT, layer=LAYER, metric_name="total_post")
scores_df = pd.DataFrame(scores_data)

sample_counts = (scores_df > 0).sum(axis=0)
sample_counts = sample_counts[sample_counts > 0]

MIN_ACTIVE_SAMPLES = 2000
live_feature_indices = sample_counts[sample_counts > MIN_ACTIVE_SAMPLES].index.tolist()

In [ ]:
def show_sample_feature_heatmap(scores_df: pd.DataFrame, feature_indices: List | np.ndarray | pd.Series, sample_indices: List | np.ndarray | pd.Series, norm: bool = True):
    sampled_scores = scores_df.iloc[sample_indices, feature_indices]

    # min-max normalization for better visualization
    if norm:
        sampled_scores = sampled_scores.apply(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0)

    plt.figure(figsize=(12, 6))
    plt.imshow(sampled_scores, aspect='auto', cmap='viridis')
    plt.colorbar(label='Feature Score')
    plt.xticks(ticks=np.arange(len(feature_indices)), labels=sampled_scores.columns, rotation=90)
    plt.yticks(ticks=np.arange(len(sample_indices)), labels=sampled_scores.index)
    plt.xlabel('Features')
    plt.ylabel('Samples')
    plt.title('Heatmap of Feature Scores for Selected Samples')
    plt.tight_layout()
    plt.show()

In [ ]:
random_features = np.random.choice(live_feature_indices, size=100, replace=False)
random_samples = np.random.choice(scores_df.index, size=50, replace=False)

In [ ]:
feature_indices_to_deactivate = random_features[10:30]
feature_indices_to_reactivate = feature_indices_to_deactivate[5:15]

In [ ]:
print("Original:")

show_sample_feature_heatmap(scores_df, random_features, random_samples)

In [ ]:
deactivated_scores_df = scores_df.copy()
deactivated_scores_df[feature_indices_to_deactivate] = 0
show_sample_feature_heatmap(deactivated_scores_df, random_features, random_samples)

In [ ]:
def show_reactivation_heatmap(scores_df: pd.DataFrame, feature_indices: List | np.ndarray | pd.Series, sample_indices: List | np.ndarray | pd.Series, features_to_react: pd.Series,  value: float = 0.9):
    sampled_scores = scores_df.iloc[sample_indices, feature_indices]

    # min-max normalization for better visualization
    sampled_scores = sampled_scores.apply(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0)

    sampled_scores[features_to_react] = value

    plt.figure(figsize=(12, 6))
    plt.imshow(sampled_scores, aspect='auto', cmap='viridis')
    plt.colorbar(label='Feature Score')
    plt.xticks(ticks=np.arange(len(feature_indices)), labels=sampled_scores.columns, rotation=90)
    plt.yticks(ticks=np.arange(len(sample_indices)), labels=sampled_scores.index)
    plt.xlabel('Features')
    plt.ylabel('Samples')
    plt.title('Heatmap of Feature Scores for Selected Samples')
    plt.tight_layout()
    plt.show()


reactivated_scores_df = deactivated_scores_df.copy()
reactivated_scores_df[feature_indices_to_reactivate] = 0
show_reactivation_heatmap(reactivated_scores_df, random_features, random_samples, feature_indices_to_reactivate, value=0.75)

### **SAEAdapter**

In [ ]:
device = torch.device("cpu")

model = AutoModelForSequenceClassification.from_pretrained(settings.lm_weights_path)
tokenizer = AutoTokenizer.from_pretrained(settings.lm_weights_path)
model.to(device)
model.eval()

model_tag = "f3072_k8"
checkpoint = f"{model_tag}_alpha"

layer = 21

feature_index = 6

sae: SparseAutoencoder = load_sae(
    model_tag=model_tag,
    hparamset_tag="alpha",
    layer_idx=layer,
    return_ckpt=False,
)
sae_adapter = SAEAdapter(model=model, sae=sae)

In [ ]:
# Load this feature's score data
scores: np.ndarray = load_score_data(checkpoint=checkpoint, layer=layer, metric_name="total_post")[:, feature_index]
scores_: np.ndarray = load_score_data(checkpoint=checkpoint, layer=layer, metric_name="avg_pre")[:, feature_index]

# Load this feature's activated sample records
sample_records: pd.DataFrame = load_sample_records(
    checkpoint=checkpoint,
    layer=layer,
    feature_index=feature_index,
    sample_type="positive",
    as_frame=True,
)

# Load text dataset for ModernBERT+adapter inferene test 
text_df = pd.read_csv(settings.text_dataset_path)

In [ ]:
feature_explanation = {
    6: {
        "best_explanation": "예시 특징",  
        "description": "사용자가 작성한 특징 설명을 입력하세요.",
    }
}

In [ ]:
def print_steering_results(
    results: Dict[str, Any], 
    text: str, 
    mode: str, 
    feature_idx: int, 
    layer: int
):
    """
    Test Helper
    """

    print("=" * 60)
    print("Text:")
    pprint(text[:500])
    print()

    print("-" * 60)
    print(f"Testing Feature: \n\t[{mode.upper()}] | Layer: {layer} | Feature: {feature_idx}")
    print(f"Explanation: {feature_explanation.get(feature_idx, {}).get('best_explanation', 'N/A')}")
    print(f"Description: {feature_explanation.get(feature_idx, {}).get('description', 'N/A')}")
    print("-" * 60)

    # 1. Feature Score 비교
    score_org = results.get('feature_score_org', 'N/A')
    score_rec = results.get('feature_score_rec', 'N/A')
    
    print("Feature Score 비교:")
    print(f"\tOriginal: {score_org:.4f}")
    print(f"\tSteered: {score_rec:.4f}")

    if mode == "deactivate":
        if isinstance(score_rec, float) and score_rec == 0.0:
            print("\t[✅] Feature 점수가 0.0으로 제거되었습니다.")
        else:
            print("\t[❌] Feature 점수가 0.0이 아닙니다.")
    elif mode == "activate":
        if isinstance(score_rec, float) and score_rec > 0.0:
            print(f"\t[✅] Feature 점수가 {score_rec:.4f}로 활성화되었습니다.")
        else:
            print("\t[❌] Feature가 활성화되지 않았습니다.")

    # 2. Logits 비교 (Softmax 확률 기준)
    logits_org = results['logits_org']
    logits_rec = results['logits_rec']
    
    probs_org = F.softmax(logits_org, dim=-1).squeeze().numpy()
    probs_rec = F.softmax(logits_rec, dim=-1).squeeze().numpy()
    
    # 클래스 개수에 맞게 포맷팅 (예: 2개 클래스)
    prob_str_org = ", ".join([f"{i}: {p:.4f}" for i, p in enumerate(probs_org)])
    prob_str_rec = ", ".join([f"{i}: {p:.4f}" for i, p in enumerate(probs_rec)])

    pred_org = 1 if probs_org[1] > probs_org[0] else 0
    pred_rec = 1 if probs_rec[1] > probs_rec[0] else 0
    icon_pred_org = "🤖" if pred_org == 1 else "👦🏻"
    icon_pred_rec = "🤖" if pred_rec == 1 else "👦🏻"

    print("-" * 60)
    print("모델 예측 비교:")
    print(f"\tOriginal: {icon_pred_org} | [{prob_str_org}]")
    print(f"\tSteered: {icon_pred_rec} | [{prob_str_rec}]")

    prob_diff = np.abs(probs_org - probs_rec).sum()
    print(f"\t[Logit Diff] {prob_diff:.6f} (L1 Norm)")
    
    if prob_diff > 1e-5:
        print("\t[✅] Feature 조정으로 인해 모델의 최종 예측이 변경되었습니다.")
    else:
        print("\t[❌] Feature 조정이 최종 예측에 큰 영향을 주지 않았습니다.")
    print("=" * 60 + "\n")

## **TEST**

In [ ]:
sample_records.head()

### (1) **deactivation**

In [ ]:
scores_.max()

In [ ]:
text = sample_records.iloc[0]["text"]
inputs = tokenizer(
    text,
    max_length=settings.prev_seq_len,
    truncation=True,
    padding="max_length",
    return_tensors="pt",
).to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

max_activation = scores_.max()

with sae_adapter.steer(
    inputs=inputs,
    layer=layer,
    feature_index=feature_index,
    mode="deactivate",
    strength=max_activation,
) as results_deact:
    
    print_steering_results(
        results=results_deact,
        text=text,
        mode="deactivate",
        feature_idx=feature_index,
        layer=layer,
    )

In [ ]:
results_deact